# Few-Shot Crop Disease Detection (Prototypical Networks)

**Scenario A: mixed-plant split.** Uses one dataset (PlantVillage). Classes are randomly
split across ALL 14 plant species into:
- **base classes** -> used to meta-train the model
- **novel classes** -> held out completely, tested later with only a few example images
  per class, possibly from plants the model never trained on at all

Run the cells top to bottom.

## 1. Install dependencies

In [ ]:
!pip install torch torchvision numpy pillow tqdm --break-system-packages -q

## 2. Download the dataset

Run this in a terminal first if you haven't already (needs a Kaggle API key configured
at `~/.kaggle/kaggle.json`):

```bash
kaggle datasets download -d abdallahalidev/plantvillage-dataset
unzip plantvillage-dataset.zip -d data/plantvillage
```

After unzipping, point `DATA_DIR` below at the folder that directly contains the
disease class subfolders (often `data/plantvillage/color` or similar — check with
`!ls data/plantvillage`).

In [ ]:
import os

DATA_DIR = "data/plantvillage/color"   # <-- update this path after you unzip
SPLIT_FILE = "split.json"
MODEL_PATH = "model.pt"

assert os.path.isdir(DATA_DIR), f"DATA_DIR not found: {DATA_DIR}. Update the path above."
print("Classes found:", len(os.listdir(DATA_DIR)))
print(sorted(os.listdir(DATA_DIR))[:5], "...")

## 3. Imports

In [ ]:
import json
import random
import argparse

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
from tqdm.notebook import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 4. Split classes into base (train) vs novel (few-shot test)

Scenario A: classes are drawn from the full mixed-plant pool, so novel classes may
belong to plant species the model has never seen at all during training.

In [ ]:
def split_classes(data_dir, novel_fraction=0.2, seed=42, out_path="split.json"):
    all_classes = sorted([
        d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))
    ])
    random.seed(seed)
    shuffled = all_classes[:]
    random.shuffle(shuffled)

    n_novel = max(1, int(len(shuffled) * novel_fraction))
    novel_classes = sorted(shuffled[:n_novel])
    base_classes = sorted(shuffled[n_novel:])

    split = {"base_classes": base_classes, "novel_classes": novel_classes}
    with open(out_path, "w") as f:
        json.dump(split, f, indent=2)

    print(f"Base (training) classes: {len(base_classes)}")
    print(f"Novel (held-out, few-shot test) classes: {len(novel_classes)}")
    print("Novel classes:", novel_classes)
    return split

split = split_classes(DATA_DIR, novel_fraction=0.2, seed=42, out_path=SPLIT_FILE)
base_classes = split["base_classes"]
novel_classes = split["novel_classes"]

## 5. Episodic dataset — samples N-way K-shot tasks

In [ ]:
IMAGE_SIZE = 84

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class EpisodicDataset:
    def __init__(self, data_dir, n_way=5, k_shot=5, q_query=15,
                 min_images_per_class=None, allowed_classes=None):
        self.data_dir = data_dir
        self.n_way = n_way
        self.k_shot = k_shot
        self.q_query = q_query

        min_required = min_images_per_class or (k_shot + q_query)
        candidate_classes = allowed_classes if allowed_classes is not None else sorted(os.listdir(data_dir))

        self.class_to_images = {}
        for class_name in candidate_classes:
            class_path = os.path.join(data_dir, class_name)
            if not os.path.isdir(class_path):
                continue
            images = [
                os.path.join(class_path, f) for f in os.listdir(class_path)
                if f.lower().endswith((".jpg", ".jpeg", ".png"))
            ]
            if len(images) >= min_required:
                self.class_to_images[class_name] = images

        self.classes = list(self.class_to_images.keys())
        if len(self.classes) < n_way:
            raise ValueError(
                f"Only {len(self.classes)} classes have >= {min_required} images, "
                f"but n_way={n_way} was requested. Lower k_shot/q_query or n_way."
            )

    def sample_episode(self):
        episode_classes = random.sample(self.classes, self.n_way)
        support_images, support_labels = [], []
        query_images, query_labels = [], []

        for label_idx, class_name in enumerate(episode_classes):
            image_paths = self.class_to_images[class_name]
            chosen = random.sample(image_paths, self.k_shot + self.q_query)
            support_paths = chosen[:self.k_shot]
            query_paths = chosen[self.k_shot:]

            for p in support_paths:
                support_images.append(self._load(p))
                support_labels.append(label_idx)
            for p in query_paths:
                query_images.append(self._load(p))
                query_labels.append(label_idx)

        return (
            torch.stack(support_images),
            torch.tensor(support_labels),
            torch.stack(query_images),
            torch.tensor(query_labels),
            episode_classes,
        )

    @staticmethod
    def _load(path):
        img = Image.open(path).convert("RGB")
        return transform(img)

## 6. Model — CNN encoder + Prototypical Network

In [ ]:
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )


class Encoder(nn.Module):
    def __init__(self, in_channels=3, hidden_channels=64, embedding_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            conv_block(in_channels, hidden_channels),
            conv_block(hidden_channels, hidden_channels),
            conv_block(hidden_channels, hidden_channels),
            conv_block(hidden_channels, embedding_dim),
        )

    def forward(self, x):
        x = self.net(x)
        return x.view(x.size(0), -1)


def euclidean_dist(a, b):
    n, m = a.size(0), b.size(0)
    a = a.unsqueeze(1).expand(n, m, -1)
    b = b.unsqueeze(0).expand(n, m, -1)
    return torch.pow(a - b, 2).sum(dim=2)


class PrototypicalNet(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder

    def forward(self, support_images, support_labels, query_images, n_way):
        support_embeddings = self.encoder(support_images)
        query_embeddings = self.encoder(query_images)

        embedding_dim = support_embeddings.size(-1)
        prototypes = torch.zeros(n_way, embedding_dim, device=support_embeddings.device)
        for c in range(n_way):
            prototypes[c] = support_embeddings[support_labels == c].mean(dim=0)

        dists = euclidean_dist(query_embeddings, prototypes)
        return F.log_softmax(-dists, dim=1)

## 7. Meta-train on the base classes

Adjust `N_WAY`, `K_SHOT`, `Q_QUERY`, `EPISODES` as needed. Start with a small
`EPISODES` value (e.g. 100) first to confirm everything runs, then increase.

In [ ]:
N_WAY = 5
K_SHOT = 20
Q_QUERY = 15
EPISODES = 2000      # increase to 10000+ for a real training run
LR = 1e-3

train_dataset = EpisodicDataset(
    data_dir=DATA_DIR, n_way=N_WAY, k_shot=K_SHOT, q_query=Q_QUERY,
    allowed_classes=base_classes,
)
print(f"Training on {len(train_dataset.classes)} usable base classes")

encoder = Encoder()
model = PrototypicalNet(encoder).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2000, gamma=0.5)

running_loss, running_acc = 0.0, 0.0
log_every = 50

for episode in tqdm(range(1, EPISODES + 1), desc="Training episodes"):
    support_x, support_y, query_x, query_y, _ = train_dataset.sample_episode()
    support_x, support_y = support_x.to(device), support_y.to(device)
    query_x, query_y = query_x.to(device), query_y.to(device)

    optimizer.zero_grad()
    log_p_y = model(support_x, support_y, query_x, N_WAY)
    loss = F.nll_loss(log_p_y, query_y)
    loss.backward()
    optimizer.step()
    scheduler.step()

    preds = log_p_y.argmax(dim=1)
    acc = (preds == query_y).float().mean().item()
    running_loss += loss.item()
    running_acc += acc

    if episode % log_every == 0:
        print(f"[Episode {episode}] loss={running_loss/log_every:.4f} acc={running_acc/log_every:.4f}")
        running_loss, running_acc = 0.0, 0.0

torch.save(model.encoder.state_dict(), MODEL_PATH)
print("Saved trained encoder to", MODEL_PATH)

## 8. Evaluate few-shot accuracy on the held-out NOVEL classes

This is your headline result — accuracy on diseases (possibly from unseen plants)
that the model never trained on.

In [ ]:
EVAL_K_SHOT = 20
EVAL_Q_QUERY = 10
EVAL_EPISODES = 200

eval_dataset = EpisodicDataset(
    data_dir=DATA_DIR, n_way=N_WAY, k_shot=EVAL_K_SHOT, q_query=EVAL_Q_QUERY,
    allowed_classes=novel_classes,
)
print(f"Evaluating on {len(eval_dataset.classes)} usable novel classes")

eval_encoder = Encoder()
eval_encoder.load_state_dict(torch.load(MODEL_PATH, map_location=device))
eval_model = PrototypicalNet(eval_encoder).to(device)
eval_model.eval()

accuracies = []
with torch.no_grad():
    for _ in tqdm(range(EVAL_EPISODES), desc="Evaluating episodes"):
        support_x, support_y, query_x, query_y, classes_used = eval_dataset.sample_episode()
        support_x, support_y = support_x.to(device), support_y.to(device)
        query_x, query_y = query_x.to(device), query_y.to(device)

        log_p_y = eval_model(support_x, support_y, query_x, N_WAY)
        preds = log_p_y.argmax(dim=1)
        acc = (preds == query_y).float().mean().item()
        accuracies.append(acc)

mean_acc = np.mean(accuracies)
ci95 = 1.96 * np.std(accuracies) / np.sqrt(len(accuracies))
print(f"\n{N_WAY}-way {EVAL_K_SHOT}-shot accuracy on held-out novel classes: "
      f"{mean_acc*100:.2f}% +/- {ci95*100:.2f}%")

## 9. Baseline comparison — plain CNN fine-tuned from scratch

Uses only the same few support images per episode, with no meta-learning.
Should score noticeably lower than the Prototypical Network above.

In [ ]:
class SimpleClassifier(nn.Module):
    def __init__(self, encoder, embedding_dim, n_way):
        super().__init__()
        self.encoder = encoder
        self.fc = nn.Linear(embedding_dim, n_way)

    def forward(self, x):
        return self.fc(self.encoder(x))


def run_baseline_episode(support_x, support_y, query_x, query_y, n_way, finetune_steps=50):
    enc = Encoder().to(device)
    with torch.no_grad():
        dummy = enc(support_x[:1])
    clf = SimpleClassifier(enc, dummy.size(-1), n_way).to(device)
    opt = torch.optim.Adam(clf.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    clf.train()
    for _ in range(finetune_steps):
        opt.zero_grad()
        loss = loss_fn(clf(support_x), support_y)
        loss.backward()
        opt.step()

    clf.eval()
    with torch.no_grad():
        preds = clf(query_x).argmax(dim=1)
        return (preds == query_y).float().mean().item()


BASELINE_EPISODES = 50   # baseline is slow (trains a tiny CNN from scratch each episode)

baseline_dataset = EpisodicDataset(
    data_dir=DATA_DIR, n_way=N_WAY, k_shot=EVAL_K_SHOT, q_query=EVAL_Q_QUERY,
    allowed_classes=novel_classes,
)

baseline_accuracies = []
for _ in tqdm(range(BASELINE_EPISODES), desc="Baseline episodes"):
    support_x, support_y, query_x, query_y, _ = baseline_dataset.sample_episode()
    support_x, support_y = support_x.to(device), support_y.to(device)
    query_x, query_y = query_x.to(device), query_y.to(device)
    acc = run_baseline_episode(support_x, support_y, query_x, query_y, N_WAY)
    baseline_accuracies.append(acc)

b_mean = np.mean(baseline_accuracies)
b_ci95 = 1.96 * np.std(baseline_accuracies) / np.sqrt(len(baseline_accuracies))
print(f"\nBaseline (from-scratch fine-tune) accuracy: {b_mean*100:.2f}% +/- {b_ci95*100:.2f}%")
print(f"Prototypical Network accuracy:               {mean_acc*100:.2f}% +/- {ci95*100:.2f}%")

## 10. Plot the comparison

A simple bar chart for your resume/report/demo.

In [ ]:
import matplotlib.pyplot as plt

labels = ["Baseline\n(fine-tuned CNN)", "Prototypical\nNetwork"]
values = [b_mean * 100, mean_acc * 100]
errors = [b_ci95 * 100, ci95 * 100]

plt.figure(figsize=(5, 4))
plt.bar(labels, values, yerr=errors, capsize=8, color=["#999999", "#4C72B0"])
plt.ylabel("Accuracy (%)")
plt.title(f"{N_WAY}-way {EVAL_K_SHOT}-shot accuracy on unseen classes")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()